<a href="https://colab.research.google.com/github/pradervonsky/vbig-lab/blob/main/evaluation/evaluation_L3-L4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Small VLMs Evaluation Pipeline with G-Eval

## Initial steps

In [20]:
!pip install --q supabase deepeval

In [21]:
import re
import math
import numpy as np
import pandas as pd
from google.colab import userdata, drive
from supabase import create_client

from deepeval.metrics import GEval
from deepeval.metrics.g_eval import Rubric
from deepeval.test_case import LLMTestCase, SingleTurnParams

In [22]:
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")
OPENAI_KEY   = userdata.get("OPENAI_API_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

import os
os.environ["OPENAI_API_KEY"] = OPENAI_KEY

### Load df_eval parquet from Drive

In [23]:
drive.mount("/content/drive")

df_eval = pd.read_parquet("/content/drive/MyDrive/df_eval.parquet")

print(f"df_eval loaded : {df_eval.shape}")
print(f"Models         : {sorted(df_eval['model_name'].unique())}")
print(f"Levels         : {df_eval['level'].unique()}")
print(f"Dashboards     : {df_eval['metadata_id'].nunique()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
df_eval loaded : (6750, 15)
Models         : ['InternVL2-1B', 'InternVL2-2B', 'InternVL3-1B-hf', 'Qwen3-VL-2B-Instruct', 'Qwen3.5-0.8B', 'Qwen3.5-2B', 'SmolVLM-256M-Instruct', 'moondream2']
Levels         : ['L2' 'L3' 'L4']
Dashboards     : 40


### Load eval_scores from Supabase

In [24]:
all_rows  = []
page_size = 1000
offset    = 0

while True:
    resp = supabase.table("eval_scores").select(
        "model_name, metadata_id, chart_id, level, "
        "rouge1, rouge2, rougel, bertscore_f1"
    ).range(offset, offset + page_size - 1).execute()
    batch = resp.data
    if not batch:
        break
    all_rows.extend(batch)
    if len(batch) < page_size:
        break
    offset += page_size

df_scores = pd.DataFrame(all_rows)
df_scores = df_scores.rename(columns={"rougel": "rougeL"})
print(f"eval_scores loaded : {df_scores.shape}")

eval_scores loaded : (6750, 8)


### Merge

In [25]:
df_qual = df_eval.merge(
    df_scores,
    on=["model_name", "metadata_id", "chart_id", "level"],
    how="left"
)

# Exclude SmolVLM — 100% degenerate (prompt-echo), no parseable output
df_qual = df_qual[df_qual["model_name"] != "SmolVLM-256M-Instruct"].copy()

print(f"df_qual shape : {df_qual.shape}")
print(f"Models        : {sorted(df_qual['model_name'].unique())}")

df_qual shape : (6297, 19)
Models        : ['InternVL2-1B', 'InternVL2-2B', 'InternVL3-1B-hf', 'Qwen3-VL-2B-Instruct', 'Qwen3.5-0.8B', 'Qwen3.5-2B', 'moondream2']


In [26]:
l3l4 = df_qual[df_qual["level"].isin(["L3","L4"])]
print(f"\nL3+L4 total  : {len(l3l4)}")
print(f"both notna   : {(l3l4['generated'].notna() & l3l4['reference'].notna()).sum()}")


L3+L4 total  : 4198
both notna   : 2012


### G-Eval Rubrics


In [27]:
# LLM-as-judge paradigm
# Evaluation criteria operationalised from Lundgard & Satyanarayan (2022)
#
# Structure per Liu et al. (2023) form-filling paradigm:
#   criteria         : short description of what is being evaluated
#   evaluation_steps : procedural steps (manual, skips auto-CoT generation)
#   rubric           : confines output to 1-5 scale with per-score descriptions
#
# Token probability weighting is handled automatically by deepeval.
# deepeval normalises the final score to [0,1].

geval_l3 = GEval(
    name="Pattern Accuracy",
    criteria=(
        "Determine whether the model-generated Level 3 insight correctly identifies "
        "the same perceptual pattern as the reference insight. A Level 3 insight "
        "describes an observable visual pattern: a trend direction or trajectory, "
        "a synthesised pattern across multiple segments or metrics, or a visible "
        "exception or outlier."
    ),
    evaluation_steps=[
        "Read the expected output and identify the pattern it describes: "
        "a trend or trajectory, a cross-segment synthesis, or a visible exception.",

        "Read the actual output and determine whether it identifies the same "
        "pattern type and whether the direction, shape, or deviation is consistent "
        "with the expected output.",

        "Assess whether the actual output describes an observable pattern using "
        "directional or descriptive language rather than reporting a specific "
        "numerical value.",
    ],
    rubric=[
        Rubric(score_range=(1, 1), expected_outcome="No valid pattern identified, or the pattern contradicts the expected output."),
        Rubric(score_range=(2, 2), expected_outcome="A pattern is mentioned but it is vague, generic, or inconsistent with the expected output."),
        Rubric(score_range=(3, 3), expected_outcome="Partially correct; a valid pattern is identified but key features are missed, misrepresented, or described too vaguely."),
        Rubric(score_range=(4, 4), expected_outcome="Same pattern type as the expected output, with minor differences in specifics such as magnitude, timing, or comparator."),
        Rubric(score_range=(5, 5), expected_outcome="Same pattern as the expected output, consistent in direction or deviation, described using specific observational language."),
    ],
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model="gpt-4o",
)

geval_l4 = GEval(
    name="Implication Quality",
    criteria=(
        "Determine whether the model-generated Level 4 insight provides a specific, "
        "grounded domain implication consistent with the reference insight. A Level 4 "
        "insight connects observed data patterns to broader business context or domain "
        "knowledge that goes beyond what is visible in the chart, such as explaining "
        "why a metric behaves as it does, why a trend occurs, or why a segment differs "
        "from others."
    ),
    evaluation_steps=[
        "Read the expected output and identify which specific observation it grounds "
        "its implication in and what domain reasoning it applies.",

        "Read the actual output and determine whether its implication is grounded "
        "in a specific observation or is generic and applicable to any dashboard.",

        "Assess whether the domain reasoning is consistent with the expected output "
        "logic, even if worded differently.",
    ],
    rubric=[
        Rubric(score_range=(1, 1), expected_outcome="No meaningful implication, off-topic, or entirely speculative without grounding."),
        Rubric(score_range=(2, 2), expected_outcome="Vague or generic implication with little grounding in chart content, or merely restates what is described in the reference."),
        Rubric(score_range=(3, 3), expected_outcome="Plausible implication but generic or weakly connected to the observed data; could apply to many dashboards."),
        Rubric(score_range=(4, 4), expected_outcome="Reasonable implication, less specific or only partially aligned with the reference logic."),
        Rubric(score_range=(5, 5), expected_outcome="Specific, grounded implication traceable to observed data, consistent with reference reasoning."),
    ],
    evaluation_params=[
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    model="gpt-4o",
)

print("GEval metrics defined.")

GEval metrics defined.


## Pilot run

In [28]:
# # Scores should be between 0.0 and 1.0.
# # Reasons should reference specific content from generated and reference texts.
# # If scores are all 0.0 or all 1.0, or reasons are generic, revisit Cell 6.

# for level in ["L3", "L4"]:
#     candidate = df_qual[
#         (df_qual["model_name"] == "Qwen3.5-2B") &
#         (df_qual["level"]      == level) &
#         df_qual["generated"].notna() &
#         df_qual["reference"].notna()
#     ].iloc[0]

#     metric = geval_l3 if level == "L3" else geval_l4

#     test_case = LLMTestCase(
#         input           = "",
#         actual_output   = str(candidate["generated"]),
#         expected_output = str(candidate["reference"]),
#     )

#     metric.measure(test_case)

#     print(f"=== Level {level} ===")
#     print(f"Dashboard  : {candidate['dashboard_name']}")
#     print(f"Chart      : {candidate['chart_id']}")
#     print(f"Generated  : {candidate['generated']}")
#     print(f"Reference  : {candidate['reference']}")
#     print(f"Score      : {metric.score:.4f}")
#     print(f"Reason     : {metric.reason}")
#     print("-" * 70)

## Full run G-Eval

In [29]:
# Runs synchronously across all 7 models.
# deepeval handles token probability weighting automatically.
# Colab session must stay alive for the full duration.
# Estimated time: ~30-60 min depending on OpenAI response times.

geval_subset = df_qual[
    df_qual["level"].isin(["L3", "L4"]) &
    df_qual["generated"].notna() &
    df_qual["reference"].notna()
].copy()

print(f"Total units to score : {len(geval_subset)}")
print(geval_subset["model_name"].value_counts().to_string())

results = []
total   = len(geval_subset)

for i, (idx, row) in enumerate(geval_subset.iterrows()):
    if i % 50 == 0:
        print(f"  Progress: {i}/{total}")

    metric = geval_l3 if row["level"] == "L3" else geval_l4

    test_case = LLMTestCase(
        input           = "",
        actual_output   = str(row["generated"]),
        expected_output = str(row["reference"]),
    )

    try:
        metric.measure(test_case)
        score = metric.score
    except Exception as e:
        print(f"  Failed at index {i} ({row['model_name']} {row['level']}): {e}")
        score = None

    results.append({
        "model_name"  : row["model_name"],
        "metadata_id" : str(row["metadata_id"]),
        "chart_id"    : int(row["chart_id"]),
        "level"       : row["level"],
        "geval"       : round(score, 4) if score is not None else None,
        "geval_reason": metric.reason if score is not None else None,
    })

df_geval_results = pd.DataFrame(results)
print(f"\nDone. {df_geval_results['geval'].notna().sum()} / {len(df_geval_results)} scored.")
print(df_geval_results.groupby(["model_name","level"])["geval"].mean().round(4).to_string())

Output()

Total units to score : 2012
model_name
Qwen3.5-0.8B            360
Qwen3.5-2B              345
Qwen3-VL-2B-Instruct    337
InternVL3-1B-hf         316
InternVL2-2B            299
InternVL2-1B            193
moondream2              162
  Progress: 0/2012


Output()

Output()

Output()

...

Output()

Output()

Output()

Output()


Done. 2012 / 2012 scored.
model_name            level
InternVL2-1B          L3       0.0511
                      L4       0.1179
InternVL2-2B          L3       0.0749
                      L4       0.2417
InternVL3-1B-hf       L3       0.1015
                      L4       0.2596
Qwen3-VL-2B-Instruct  L3       0.2691
                      L4       0.3892
Qwen3.5-0.8B          L3       0.1022
                      L4       0.2742
Qwen3.5-2B            L3       0.2750
                      L4       0.4245
moondream2            L3       0.0258
                      L4       0.1467


### Upsert G-Eval to Supabase

In [30]:
GEVAL_STORE_COLS = [
    "model_name", "metadata_id", "chart_id", "level",
    "geval", "geval_reason",
]

df_upload = df_geval_results[GEVAL_STORE_COLS].copy()
df_upload = df_upload.drop_duplicates(
    subset=["model_name", "metadata_id", "chart_id", "level"],
    keep="last"
)
df_upload["metadata_id"] = df_upload["metadata_id"].astype(str)
df_upload["chart_id"]    = df_upload["chart_id"].astype(int)

records = (
    df_upload
    .replace({float("nan"): None, float("inf"): None, float("-inf"): None})
    .to_dict(orient="records")
)

response = supabase.table("eval_scores").upsert(
    records,
    on_conflict="model_name,metadata_id,chart_id,level"
).execute()

print(f"Upserted {len(records)} G-Eval rows to eval_scores.")
print("ROUGE and BERTScore columns untouched.")

Upserted 2012 G-Eval rows to eval_scores.
ROUGE and BERTScore columns untouched.


## Qualitative analysis

In [31]:
# Reload Supabase for qualitative analysis
all_rows  = []
page_size = 1000
offset    = 0

while True:
    resp = supabase.table("eval_scores").select(
        "model_name, metadata_id, chart_id, level, "
        "rouge1, rouge2, rougel, bertscore_f1, "
        "geval, geval_reason"
    ).range(offset, offset + page_size - 1).execute()
    batch = resp.data
    if not batch:
        break
    all_rows.extend(batch)
    if len(batch) < page_size:
        break
    offset += page_size

df_scores_full = pd.DataFrame(all_rows)
df_scores_full = df_scores_full.rename(columns={"rougel": "rougeL"})

df_qual = df_qual.drop(
    columns=["rouge1","rouge2","rougeL","bertscore_f1",
             "geval","geval_reason"],
    errors="ignore"
).merge(
    df_scores_full,
    on=["model_name","metadata_id","chart_id","level"],
    how="left"
)

geval_scored = df_qual["geval"].notna().sum()
l3l4_total   = len(df_qual[df_qual["level"].isin(["L3","L4"])])
print(f"G-Eval coverage : {geval_scored} / {l3l4_total} L3+L4 pairs scored")

G-Eval coverage : 2012 / 4198 L3+L4 pairs scored


### Hardest dashboards

In [41]:
# Per dashboard: average score across all models and all levels.
# Composite uses rougeL, bertscore_f1, and geval.

def row_composite(r):
    vals = []
    if pd.notna(r.get("rougeL")):       vals.append(r["rougeL"])
    if pd.notna(r.get("bertscore_f1")): vals.append(r["bertscore_f1"])
    if pd.notna(r.get("geval")):   vals.append(r["geval"])
    return np.mean(vals) if vals else np.nan

df_qual["composite"] = df_qual.apply(row_composite, axis=1)

dashboard_difficulty = (
    df_qual.groupby(["metadata_id","dashboard_name"])["composite"]
    .mean()
    .reset_index()
    .rename(columns={"composite": "avg_composite_score"})
    .sort_values("avg_composite_score")
)

meta_resp = supabase.table("metadata").select("id, dashboard_author").execute()
df_meta_full = pd.DataFrame(meta_resp.data).rename(columns={"id": "metadata_id"})

dashboard_difficulty = dashboard_difficulty.merge(
    df_meta_full[["metadata_id","dashboard_author"]],
    on="metadata_id",
    how="left"
)

N_HARD = 5
display_cols = ["dashboard_name","dashboard_author","avg_composite_score"]

print(f"=== Hardest Dashboards ===")
print(dashboard_difficulty[display_cols].head(N_HARD).round(4).to_string(index=False))
print(f"\n=== Easiest Dashboards ===")
print(dashboard_difficulty[display_cols].tail(N_HARD).round(4).to_string(index=False))

=== Hardest Dashboards ===
                                        dashboard_name dashboard_author  avg_composite_score
                       Superstore Performance Overview         Julie Li               0.1397
                         Superstore Dashboard Overview        Cathy Lau               0.1433
              Superstore Dashboard - Business Overview    Murilo Cremon               0.1606
                             Superstore Sales Overview     Keren Aharon               0.1681
Superstore Order Details | KPI's and Selection Filters   Naresh Suglani               0.1711

=== Easiest Dashboards ===
                                   dashboard_name dashboard_author  avg_composite_score
                                  Superstore KPIs     Andy Kriebel               0.2667
                 Superstore Sales Dashboard #VOTD    Israel Ayoola               0.2767
                                       Superstore        Bill Yost               0.2784
SuperStore Dashboard - 2019 Sales &

### Top scoring samples

In [45]:
TOP_N = 1

# Load author lookup once
meta_resp = supabase.table("metadata").select("id, dashboard_author").execute()
df_meta_full = pd.DataFrame(meta_resp.data).rename(columns={"id": "metadata_id"})

top_examples = []
for model in sorted(df_qual["model_name"].unique()):
    for level in ["L2", "L3", "L4"]:
        if level == "L4":
            metric = "geval"
        elif level == "L2":
            metric = "bertscore_f1"
        else:  # L3
            metric = "bertscore_f1"
        sub = df_qual[
            (df_qual["model_name"] == model) &
            (df_qual["level"]      == level) &
            df_qual[metric].notna()
        ].nlargest(TOP_N, metric)

        for _, r in sub.iterrows():
            top_examples.append({
                "model"            : model,
                "level"            : level,
                "dashboard_name"   : r["dashboard_name"],
                "metadata_id"      : r["metadata_id"],
                "chart_id"         : r["chart_id"],
                "score"            : round(r[metric], 4),
                "metric_used"      : metric,
                "generated"        : r["generated"],
                "reference"        : r["reference"],
            })

df_top = pd.DataFrame(top_examples).merge(
    df_meta_full[["metadata_id","dashboard_author"]],
    on="metadata_id",
    how="left"
)

print(f"Top examples collected: {len(df_top)}")
print(df_top[["model","level","dashboard_name","dashboard_author","chart_id","score","metric_used"]].head(21).to_string(index=False))

Top examples collected: 21
               model level                                     dashboard_name       dashboard_author  chart_id  score  metric_used
        InternVL2-1B    L2                               Superstore Dashboard          Harshit Gupta         3 0.5397 bertscore_f1
        InternVL2-1B    L3  SuperStore Dashboard - 2019 Sales & Profitability              Linh Pham         2 0.4519 bertscore_f1
        InternVL2-1B    L4                               Superstore Dashboard     Divas Pratap Singh         4 0.6976        geval
        InternVL2-2B    L2                          Superstore Sales Overview           Keren Aharon         1 0.6540 bertscore_f1
        InternVL2-2B    L3 Demo Excecutive Performance Dashboard - Superstore                An Tran         1 0.4696 bertscore_f1
        InternVL2-2B    L4                   Superstore Performance Dashboard         Tanya Lomskaya         5 0.7514        geval
     InternVL3-1B-hf    L2                              

### Low scoring samples

In [60]:
# Low scoring examples — L3 and L4 only, 1 per model per level
BOTTOM_N = 1

meta_resp = supabase.table("metadata").select("id, dashboard_author").execute()
df_meta_full = pd.DataFrame(meta_resp.data).rename(columns={"id": "metadata_id"})

bottom_examples = []

for model in sorted(df_qual["model_name"].unique()):
    for level in ["L3", "L4"]:
        metric = "geval"

        sub = df_qual[
            (df_qual["model_name"] == model) &
            (df_qual["level"]      == level) &
            df_qual[metric].notna()
        ]

        for _, r in sub.nsmallest(BOTTOM_N, metric).iterrows():
            bottom_examples.append({
                "model"         : model,
                "level"         : level,
                "dashboard_name": r["dashboard_name"],
                "metadata_id"   : r["metadata_id"],
                "chart_id"      : r["chart_id"],
                "score"         : round(r[metric], 4),
                "generated"     : r["generated"],
                "reference"     : r["reference"],
                "geval_reason"  : r.get("geval_reason", None),
            })

df_bottom = pd.DataFrame(bottom_examples).merge(
    df_meta_full[["metadata_id","dashboard_author"]], on="metadata_id", how="left"
)

print(f"=== Bottom scoring examples per model for L3 and L4 ===")
print(df_bottom[["model","level","dashboard_name","dashboard_author",
                 "chart_id","score","geval_reason"]].to_string(index=False))

=== Bottom scoring examples per model for L3 and L4 ===
               model level                              dashboard_name  dashboard_author  chart_id  score                                                                                                                                                                                                                                                                                                                                                                                  geval_reason
        InternVL2-1B    L3  Interactive Drill Down Superstore Overview      Juliet Craig         1 0.0000                                                                           The actual output identifies a ranking of states by sales revenue, which is a cross-segment synthesis, while the expected output describes a trend over time, specifically higher sales at the end of the year. The pattern types do not match, and the actual output does not addre

### Failure mode taxonomy

In [63]:
# Classify each VLM output into a failure mode
# Applied to all rows regardless of level

def classify_failure(row):
    gen = row.get("generated")
    if gen is None or (isinstance(gen, float) and math.isnan(gen)):
        return "unparseable_or_empty"
    gen_lower   = str(gen).lower()
    ref         = row.get("reference")
    ref_is_null = ref is None or (isinstance(ref, float) and math.isnan(ref))
    GENERIC_L4 = [
        "it is important to", "organizations should", "decision makers",
        "stakeholders", "further investigation", "may indicate",
    ]
    if row["level"] == "L4" and any(s in gen_lower for s in GENERIC_L4):
        return "generic_L4"
    if not ref_is_null:
        gen_nums = set(re.findall(r"\b\d[\d,.]*\b", str(gen)))
        ref_nums = set(re.findall(r"\b\d[\d,.]*\b", str(ref)))
        if len(gen_nums - ref_nums) >= 3:
            return "possible_hallucination"
    return "no_issue_detected"

df_qual["failure_mode"] = df_qual.apply(classify_failure, axis=1)

failure_summary = (
    df_qual.groupby(["model_name", "failure_mode"])
    .size()
    .reset_index(name="count")
    .pivot(index="model_name", columns="failure_mode", values="count")
    .fillna(0).astype(int)
)

failure_summary["total"] = failure_summary.sum(axis=1)

print("=== Failure Mode Taxonomy ===")
print(failure_summary.to_string())

=== Failure Mode Taxonomy ===
failure_mode          generic_L4  no_issue_detected  possible_hallucination  unparseable_or_empty  total
model_name                                                                                              
InternVL2-1B                   0               1080                      18                   768   1866
InternVL2-2B                   0                671                      45                     4    720
InternVL3-1B-hf                1                927                      83                    48   1059
Qwen3-VL-2B-Instruct          18                502                      81                     2    603
Qwen3.5-0.8B                   8               1008                      29                    14   1059
Qwen3.5-2B                     0                569                      49                     0    618
moondream2                     0                335                      11                    26    372


### Side-by-side qualitative comparison

In [35]:
def show_example(model, dashboard_name, chart_id, level):
    row = df_qual[
        (df_qual["model_name"]     == model) &
        (df_qual["dashboard_name"] == dashboard_name) &
        (df_qual["chart_id"]       == chart_id) &
        (df_qual["level"]          == level)
    ]
    if row.empty:
        print("No matching row found.")
        return
    r = row.iloc[0]
    print(f"Model      : {r['model_name']}")
    print(f"Dashboard  : {r['dashboard_name']}")
    print(f"Chart      : {r['chart_id']} - {r.get('chart_title','')}")
    print(f"Level      : {r['level']}")
    print(f"\nGenerated  :\n  {r['generated']}")
    print(f"\nReference  :\n  {r['reference']}")
    print(f"\nROUGE-1    : {round(r['rouge1'], 4) if pd.notna(r.get('rouge1')) else '-'}")
    print(f"ROUGE-2    : {round(r['rouge2'], 4) if pd.notna(r.get('rouge2')) else '-'}")
    print(f"ROUGE-L    : {round(r['rougeL'], 4) if pd.notna(r.get('rougeL')) else '-'}")
    print(f"BERTScore  : {round(r['bertscore_f1'], 4) if pd.notna(r.get('bertscore_f1')) else '-'}")
    print(f"G-Eval     : {r.get('geval','-')}")
    print(f"Failure    : {r.get('failure_mode','-')}")
    print("-" * 70)

# Example - call with any row from df_top
show_example(
    model          = df_top.iloc[0]["model"],
    dashboard_name = df_top.iloc[0]["dashboard_name"],
    chart_id       = df_top.iloc[0]["chart_id"],
    level          = df_top.iloc[0]["level"],
)

Model      : InternVL2-1B
Dashboard  : The Popular Superstore Sales Overview
Chart      : 4 - Sales by Customer
Level      : L2

Generated  :
  Raymond Buch: $14.2K

Reference  :
  The first highest sales by customer is Raymond Buch at $14.2K

ROUGE-1    : 0.5
ROUGE-2    : 0.2857
ROUGE-L    : 0.5
BERTScore  : 0.5003
G-Eval     : nan
Failure    : none
----------------------------------------------------------------------
